In [1]:
# 1. Importaciones y configuración
import os
import gc
import re
import threading
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Rutas de archivos y carpetas
BASE_DIR = "/home/g5/reto_1_2-2627-Rag-Benchmark"
CSV_ORIGEN = f"{BASE_DIR}/Benchmarks/benchmark_embedding.csv"
CSV_RESULTADOS = f"{BASE_DIR}/Benchmarks/resultados_chatbots.csv"
CHROMA_PATH = f"{BASE_DIR}/Benchmarks/data/chroma"
EMBEDDING_PATH = f"{BASE_DIR}/Modelos/Embedding/embeddinggemma-300m"
CARPETA_MODELOS = f"{BASE_DIR}/Modelos/Chat"

def ruta_modelo(nombre):
    """Devuelve la ruta al archivo .gguf del modelo."""
    ruta = f"{CARPETA_MODELOS}/{nombre}"
    if os.path.isdir(ruta):
        for archivo in os.listdir(ruta):
            if archivo.endswith(".gguf"):
                return f"{ruta}/{archivo}"
    return ruta

def responder(llm, nombre, prompt, max_tokens=256):
    """Genera respuesta adaptándose al formato de cada modelo."""
    if "K2" in nombre:
        p = f"<|ifm|im_start|>user\n{prompt}<|ifm|im_end|>\n<|ifm|im_start|>assistant\n<ifm|think>\n</ifm|think>\n"
        toks = llm.tokenize(p.encode(), special=True)
        res = llm(toks, max_tokens=max_tokens, temperature=0.0, stop=["<|ifm|im_end|>"])["choices"][0]["text"]
    elif "Ornith" in nombre:
        p = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
        toks = llm.tokenize(p.encode(), special=True)
        res = llm(toks, max_tokens=max_tokens, temperature=0.0, stop=["<|im_end|>"])["choices"][0]["text"]
    else:
        msgs = [{"role": "system", "content": "detailed thinking off"}, {"role": "user", "content": prompt}] if "Ling" in nombre else [{"role": "user", "content": prompt}]
        res = llm.create_chat_completion(messages=msgs, max_tokens=max_tokens, temperature=0.0)["choices"][0]["message"]["content"]
    return re.sub(r"<(ifm\|think|think)>.*?</\1>", "", res, flags=re.DOTALL).strip()


/home/g5/reto_1_2-2627-Rag-Benchmark/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Descargar modelos de Hugging Face
modelos = {
    "unsloth/NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF": "NVIDIA-Nemotron-3.5-Bolt-E4B-UD-Q4_K_XL.gguf",
    "IFM/K2-Horizon-3.7B-GGUF": "K2-Horizon-4B-Q6_K.gguf",
    "inclusionAI/Ling-3.0-tiny-GGUF": "Ling-3.0-tiny-Q5_K_M.gguf",
    "ornith-ai/Ornith-1.5-9B-GGUF": "Ornith-1.5-9B-Q4_K_M.gguf"
}

for repo, archivo in modelos.items():
    nombre = repo.split("/")[-1]
    destino = f"{CARPETA_MODELOS}/{nombre}"
    os.makedirs(destino, exist_ok=True)
    
    if any(f.endswith(".gguf") for f in os.listdir(destino)):
        print(f"{nombre}: ya descargado.")
    else:
        print(f"Descargando {nombre}...")
        hf_hub_download(repo, filename=archivo, local_dir=destino)


NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF: ya descargado.
K2-Horizon-3.7B-GGUF: ya descargado.
Ling-3.0-tiny-GGUF: ya descargado.
Ornith-1.5-9B-GGUF: ya descargado.


In [3]:
# 3. Cargar o crear el archivo de resultados
if os.path.exists(CSV_RESULTADOS):
    df = pd.read_csv(CSV_RESULTADOS, dtype=str)
else:
    df = pd.read_csv(CSV_ORIGEN, dtype=str)[["texto", "pregunta"]]
    df.to_csv(CSV_RESULTADOS, index=False)

df.head()

,texto,pregunta,contexto_top3,NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,K2-Horizon-3.7B-GGUF,Ling-3.0-tiny-GGUF,Ornith-1.5-9B-GGUF,evaluacion_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF_por_K2-Horizon-3.7B-GGUF,evaluacion_K2-Horizon-3.7B-GGUF_por_K2-Horizon-3.7B-GGUF,evaluacion_Ling-3.0-tiny-GGUF_por_K2-Horizon-3.7B-GGUF,...,evaluacion_Ling-3.0-tiny-GGUF_por_Ling-3.0-tiny-GGUF,evaluacion_Ornith-1.5-9B-GGUF_por_Ling-3.0-tiny-GGUF,evaluacion_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_K2-Horizon-3.7B-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_Ling-3.0-tiny-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_Ornith-1.5-9B-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF_por_Ornith-1.5-9B-GGUF,evaluacion_K2-Horizon-3.7B-GGUF_por_Ornith-1.5-9B-GGUF,evaluacion_Ling-3.0-tiny-GGUF_por_Ornith-1.5-9B-GGUF,evaluacion_Ornith-1.5-9B-GGUF_por_Ornith-1.5-9B-GGUF
0,c) El de los trabajadores después de haberse r...,¿Qué ocurre con el contrato de trabajo si el d...,5. La calificación por la autoridad judicial d...,convalidará la extinción del contrato de traba...,El despido procedente convalidará la extinción...,Se concluye la extinción del contrato de traba...,El despido procedente convalidará la extinción...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...",...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci..."
1,Contra las decisiones a que se refiere el pres...,¿Qué tipo de acción se podrá reclamar en confl...,Contra las decisiones a que se refiere el pres...,Se podrá reclamar en conflicto colectivo cuand...,No aparece en el documento,Se podrá reclamar en conflicto colectivo la ac...,No aparece en el documento.\n\nEl contexto ind...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 0, ""fidelidad"": ""no"", ""abstenci...",...,"{""correccion"": 0, ""fidelidad"": ""no"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 0, ""fidelidad"": ""no"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci..."
2,1. Los convenios colectivos tendrán el ámbito ...,¿Qué tipo de acuerdos pueden pactarse por las ...,1. Los convenios colectivos tendrán el ámbito ...,cláusulas sobre la estructura de la negociació...,Las organizaciones sindicales y asociaciones e...,Las organizaciones sindicales y asociaciones e...,Las organizaciones sindicales y asociaciones e...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...",...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion""

In [4]:
# 4. Obtener el contexto de cada pregunta usando ChromaDB
if "contexto_top3" not in df.columns or df["contexto_top3"].isna().any():
    modelo_embed = SentenceTransformer(EMBEDDING_PATH, device="cpu")
    bd = chromadb.PersistentClient(path=CHROMA_PATH).get_collection("boe_embeddinggemma-300m")
    
    consultas = ["task: search result | query: " + str(q) for q in df["pregunta"]]
    vectores = modelo_embed.encode(consultas, batch_size=64, show_progress_bar=True)
    
    contextos = []
    for vec in vectores:
        docs = bd.query(query_embeddings=[vec.tolist()], n_results=3)["documents"][0]
        contextos.append("---".join(docs))
        
    df["contexto_top3"] = contextos
    df.to_csv(CSV_RESULTADOS, index=False)

In [5]:
# 5. Generar respuestas con cada modelo de chat (2 modelos en paralelo)
prompt_base = """Basándote exclusivamente en el siguiente contexto, responde a la pregunta de forma concisa. Si la respuesta no aparece en el contexto, di exactamente: "No aparece en el documento".

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:"""

# Modelos en la carpeta
modelos_chat = sorted([m for m in os.listdir(CARPETA_MODELOS) if os.path.isdir(f"{CARPETA_MODELOS}/{m}") and not m.startswith(".")])
bloqueo_guardado = threading.Lock()

def procesar_modelo(nombre):
    if nombre not in df.columns:
        df[nombre] = None
    df[nombre] = df[nombre].astype(object)
        
    pendientes = df[df[nombre].isna() | (df[nombre] == "")].index
    if len(pendientes) == 0:
        print(f"{nombre}: ya completado.")
        return
        
    print(f"\n--- Iniciando {nombre} ({len(pendientes)} pendientes) ---")
    llm = Llama(model_path=ruta_modelo(nombre), n_gpu_layers=-1, n_ctx=2048, n_batch=2048, flash_attn=True, verbose=False)
    
    for i in pendientes:
        prompt = prompt_base.format(contexto=df.at[i, "contexto_top3"], pregunta=df.at[i, "pregunta"])
        res = responder(llm, nombre, prompt, max_tokens=256)
        with bloqueo_guardado:
            df.at[i, nombre] = res
            df.to_csv(CSV_RESULTADOS, index=False)
        print(f"[{nombre}] Fila {i} guardada.", flush=True)
        
    del llm
    gc.collect()
    print(f"{nombre}: terminado.")

# Ejecutar 2 modelos simultáneamente en la GPU
with ThreadPoolExecutor(max_workers=2) as executor:
    list(executor.map(procesar_modelo, modelos_chat))


K2-Horizon-3.7B-GGUF: ya completado.
Ling-3.0-tiny-GGUF: ya completado.
NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF: ya completado.
Ornith-1.5-9B-GGUF: ya completado.


# 6. Evaluación de las respuestas

In [6]:
# 7. Configuración de la evaluación
prompt_evaluacion = """Basándote en el contexto y la pregunta, evalúa la respuesta generada por una inteligencia artificial.

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta a evaluar: {respuesta}

Evalúa los siguientes criterios:
- correccion: 2 (si es correcta y completa, o si no aparece en el contexto y dice "No aparece en el documento"), 1 (si es incompleta o imprecisa), 0 (si es incorrecta).
- fidelidad: "si" (si no inventa datos y todo lo afirmado está en el contexto) o "no" (si alucina o inventa datos).
- abstencion: "si" (si responde diciendo que no aparece en el documento) o "no" (si responde a la pregunta).
- formato_e_idioma: "si" (si está en español y cita la fuente o artículo cuando procede) o "no".

Devuelve únicamente un JSON con este formato exacto:
{{"correccion": 2, "fidelidad": "si", "abstencion": "no", "formato_e_idioma": "si"}}"""

# Modelos evaluadores (los que estén en la carpeta)
evaluadores = sorted([m for m in os.listdir(CARPETA_MODELOS) if os.path.isdir(f"{CARPETA_MODELOS}/{m}") and not m.startswith(".")])

# Columnas con respuestas a evaluar
columnas_fijas = ["texto", "pregunta", "contexto_top3"]
modelos_a_evaluar = [col for col in df.columns if col not in columnas_fijas and not col.startswith("evaluacion_")]

print("Evaluadores:", evaluadores)
print("Modelos a evaluar:", modelos_a_evaluar)

Evaluadores: ['K2-Horizon-3.7B-GGUF', 'Ling-3.0-tiny-GGUF', 'NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF', 'Ornith-1.5-9B-GGUF']
Modelos a evaluar: ['NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF', 'K2-Horizon-3.7B-GGUF', 'Ling-3.0-tiny-GGUF', 'Ornith-1.5-9B-GGUF']


In [7]:
# 8. Ejecutar la evaluación y guardar en el CSV (2 evaluadores en paralelo)
bloqueo_guardado = threading.Lock()

def procesar_evaluador(evaluador):
    llm = None
    for modelo in modelos_a_evaluar:
        columna_eval = f"evaluacion_{modelo}_por_{evaluador}"
        if columna_eval not in df.columns:
            df[columna_eval] = None
        df[columna_eval] = df[columna_eval].astype(object)
            
        pendientes = df[df[modelo].notna() & (df[modelo] != "") & (df[columna_eval].isna() | (df[columna_eval] == ""))].index
        if len(pendientes) == 0:
            continue
            
        if llm is None:
            print(f"\n--- Cargando evaluador: {evaluador} ---")
            llm = Llama(model_path=ruta_modelo(evaluador), n_gpu_layers=-1, n_ctx=2048, n_batch=2048, flash_attn=True, verbose=False)
                
        print(f"[{evaluador}] Evaluando respuestas de {modelo} ({len(pendientes)} pendientes)...")
        for i in pendientes:
            prompt = prompt_evaluacion.format(
                contexto=df.at[i, "contexto_top3"],
                pregunta=df.at[i, "pregunta"],
                respuesta=df.at[i, modelo]
            )
            res = responder(llm, evaluador, prompt, max_tokens=128)
            with bloqueo_guardado:
                df.at[i, columna_eval] = res
                df.to_csv(CSV_RESULTADOS, index=False)
            print(f"[{evaluador}] Evaluada fila {i} de {modelo}.", flush=True)
            
    if llm is not None:
        del llm
        gc.collect()
        print(f"Evaluador {evaluador} finalizado.")

# Ejecutar 2 evaluadores simultáneamente en la GPU
with ThreadPoolExecutor(max_workers=2) as executor:
    list(executor.map(procesar_evaluador, evaluadores))

print("\n¡Evaluación completada!")



¡Evaluación completada!


In [8]:
# 9. Ver los resultados guardados en el DataFrame
df.head()

,texto,pregunta,contexto_top3,NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,K2-Horizon-3.7B-GGUF,Ling-3.0-tiny-GGUF,Ornith-1.5-9B-GGUF,evaluacion_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF_por_K2-Horizon-3.7B-GGUF,evaluacion_K2-Horizon-3.7B-GGUF_por_K2-Horizon-3.7B-GGUF,evaluacion_Ling-3.0-tiny-GGUF_por_K2-Horizon-3.7B-GGUF,...,evaluacion_Ling-3.0-tiny-GGUF_por_Ling-3.0-tiny-GGUF,evaluacion_Ornith-1.5-9B-GGUF_por_Ling-3.0-tiny-GGUF,evaluacion_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_K2-Horizon-3.7B-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_Ling-3.0-tiny-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_Ornith-1.5-9B-GGUF_por_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF,evaluacion_NVIDIA-Nemotron-3.5-Bolt-E4B-GGUF_por_Ornith-1.5-9B-GGUF,evaluacion_K2-Horizon-3.7B-GGUF_por_Ornith-1.5-9B-GGUF,evaluacion_Ling-3.0-tiny-GGUF_por_Ornith-1.5-9B-GGUF,evaluacion_Ornith-1.5-9B-GGUF_por_Ornith-1.5-9B-GGUF
0,c) El de los trabajadores después de haberse r...,¿Qué ocurre con el contrato de trabajo si el d...,5. La calificación por la autoridad judicial d...,convalidará la extinción del contrato de traba...,El despido procedente convalidará la extinción...,Se concluye la extinción del contrato de traba...,El despido procedente convalidará la extinción...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...",...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci..."
1,Contra las decisiones a que se refiere el pres...,¿Qué tipo de acción se podrá reclamar en confl...,Contra las decisiones a que se refiere el pres...,Se podrá reclamar en conflicto colectivo cuand...,No aparece en el documento,Se podrá reclamar en conflicto colectivo la ac...,No aparece en el documento.\n\nEl contexto ind...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 0, ""fidelidad"": ""no"", ""abstenci...",...,"{""correccion"": 0, ""fidelidad"": ""no"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 0, ""fidelidad"": ""no"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci..."
2,1. Los convenios colectivos tendrán el ámbito ...,¿Qué tipo de acuerdos pueden pactarse por las ...,1. Los convenios colectivos tendrán el ámbito ...,cláusulas sobre la estructura de la negociació...,Las organizaciones sindicales y asociaciones e...,Las organizaciones sindicales y asociaciones e...,Las organizaciones sindicales y asociaciones e...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...",...,"{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 1, ""fidelidad"": ""si"", ""abstenci...","{""correccion"": 2, ""fidelidad"": ""si"", ""abstenci...","{""correccion""